# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_07 — Beta-Weighted Tail-Risk Hedging

### Research question

How do we hedge a risky multi-asset portfolio using beta-weighted exposure, downside beta, futures overlays, and convex tail hedges, while measuring the cost and effectiveness of each hedge?

This notebook follows:

```text
04_02_capm_and_performance_attribution
04_03_volatility_targeting_and_position_sizing
04_05_pca_spectral_risk_analysis
04_06_var_cvar_and_stress_testing
```

The previous notebook measured tail risk using VaR, CVaR, and stress testing. This notebook asks:

> Once we know the portfolio has downside risk, how do we hedge it?

It covers:

1. market beta estimation;
2. beta-weighted exposure;
3. rolling beta;
4. downside beta / tail beta;
5. market futures hedge ratio;
6. partial beta hedging;
7. basis risk;
8. hedge cost and carry;
9. convex put-style tail hedge;
10. collar-style hedge;
11. dynamic hedge overlays;
12. VaR/CVaR before and after hedging;
13. stress scenario hedge effectiveness;
14. drawdown impact;
15. beta slippage in crisis regimes;
16. hedge attribution;
17. governance checks;
18. saved outputs and manifest.

The key idea:

> Beta hedging reduces linear market exposure. Tail-risk hedging tries to reduce crash losses. They overlap, but they are not the same thing.

## 1. Beta-weighted exposure

For asset return $r_i$ and market return $r_m$, beta is:

$$
\beta_i = \frac{Cov(r_i,r_m)}{Var(r_m)}
$$

Portfolio beta:

$$
\beta_p = \sum_i w_i\beta_i
$$

If portfolio value is $V$, beta-weighted market exposure is:

$$
BWE = V\beta_p
$$

A portfolio with $V=\$1,000,000$ and $\beta_p=0.80$ has roughly:

$$
\$800,000
$$

of market-equivalent exposure.

## 2. Futures hedge ratio

If the hedge instrument has beta $\beta_h$, the hedge weight required to target beta $\beta^*$ is:

$$
w_h = \frac{\beta^*-\beta_p}{\beta_h}
$$

If $\beta^*=0$ and $\beta_h \approx 1$, then:

$$
w_h\approx-\beta_p
$$

In dollar notional terms:

$$
Notional_h = w_h V
$$

For futures contracts:

$$
contracts = \frac{Notional_h} {F \times multiplier}
$$

This notebook works mostly in return-weight space, then shows a contract-sizing example.

## 3. Downside beta and tail beta

Standard beta uses all observations.

Downside beta uses only bad market days:

$$
\beta_i^{down} = \frac{Cov(r_i,r_m \mid r_m < q_\alpha)} {Var(r_m \mid r_m < q_\alpha)}
$$

This matters because:

- correlations often rise in stress;
- defensive assets may behave differently in crashes;
- a normal beta hedge may under-hedge tail beta;
- hedge instruments may also change beta in stress.

A robust hedge process should compare normal beta and downside beta.

## 4. Convex tail hedging

Short futures hedge linearly:

$$
PnL_h \approx w_h r_m
$$

A put-style hedge has convex payoff:

$$
Payoff = \max(K-S_T,0)
$$

Convex hedges can protect against large losses, but they cost premium and can drag performance in calm markets.

This notebook uses a simplified one-period rolling put proxy to demonstrate the mechanics. It is not an options pricing engine.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from statistics import NormalDist
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class BetaTailHedgeConfig:
    n_days: int = 2_400
    annualisation: int = 252
    rolling_beta_window: int = 126
    downside_quantile: float = 0.10
    target_beta_full_hedge: float = 0.0
    target_beta_partial_hedge: float = 0.35
    initial_capital: float = 1_000_000.0
    transaction_cost_bps_futures: float = 1.5
    hedge_carry_bps_ann: float = 35.0
    put_maturity_days: int = 21
    put_moneyness: float = 0.95
    put_budget_ann: float = 0.015
    collar_call_moneyness: float = 1.07
    stress_sigma: float = 3.0
    seed: int = 42


config = BetaTailHedgeConfig()
config

## 6. Simulate market, portfolio assets, and hedge instruments

We simulate:

### Risk assets

- equities;
- credit;
- real estate;
- commodities;
- crypto proxy;
- trend-following diversifier.

### Hedge instruments

- equity index future;
- treasury future;
- gold future;
- VIX / convexity proxy;
- rolling put proxy.

The simulation includes:

1. calm and crisis regimes;
2. market crashes;
3. correlation spikes;
4. basis risk between portfolio and hedge future;
5. convex payoff behaviour.

In [ ]:
def simulate_beta_tail_hedge_universe(config: BetaTailHedgeConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2014-01-01", periods=config.n_days)

    asset_cols = [
        "US_EQ", "EU_EQ", "EM_EQ",
        "CREDIT", "REIT",
        "OIL", "COPPER", "GOLD",
        "FX_CARRY", "TREND",
        "BTC_PROXY"
    ]

    hedge_cols = [
        "INDEX_FUT",
        "TREASURY_FUT",
        "GOLD_FUT",
        "VIX_PROXY"
    ]

    all_cols = asset_cols + hedge_cols + ["MARKET"]

    # Regime process.
    regime = np.zeros(config.n_days, dtype=int)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    market = np.zeros(config.n_days)
    rates = np.zeros(config.n_days)
    commodity = np.zeros(config.n_days)
    volatility_factor = np.zeros(config.n_days)
    trend_factor = np.zeros(config.n_days)

    stress_type = np.array(["normal"] * config.n_days, dtype=object)

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_mult = 1.0 if regime[t] == 0 else 2.5

        market[t] = 0.00025 + vol_mult * 0.010 * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)
        rates[t] = 0.00005 + vol_mult * 0.0035 * rng.standard_t(df=7) * np.sqrt((7 - 2) / 7)
        commodity[t] = 0.00010 + vol_mult * 0.0090 * rng.standard_t(df=6) * np.sqrt((6 - 2) / 6)
        trend_factor[t] = 0.00015 + vol_mult * 0.0060 * rng.standard_t(df=8) * np.sqrt((8 - 2) / 8)
        volatility_factor[t] = -0.20 * market[t] + vol_mult * 0.006 * rng.standard_normal()

        u = rng.uniform()
        if u < 0.010:
            stress_type[t] = "equity_crash"
            shock = rng.uniform(0.035, 0.105)
            market[t] -= shock
            rates[t] += rng.uniform(0.003, 0.018)
            volatility_factor[t] += rng.uniform(0.040, 0.120)
            commodity[t] += rng.normal(0.0, 0.030)
            trend_factor[t] += rng.uniform(0.005, 0.035)
        elif u < 0.016:
            stress_type[t] = "inflation_shock"
            market[t] -= rng.uniform(0.010, 0.050)
            rates[t] -= rng.uniform(0.010, 0.035)
            commodity[t] += rng.uniform(0.030, 0.090)
            volatility_factor[t] += rng.uniform(0.010, 0.050)
        elif u < 0.022:
            stress_type[t] = "commodity_crash"
            commodity[t] -= rng.uniform(0.040, 0.120)
            market[t] -= rng.uniform(0.005, 0.030)
            volatility_factor[t] += rng.uniform(0.005, 0.035)
        elif u < 0.027:
            stress_type[t] = "crypto_crash"
            market[t] -= rng.uniform(0.005, 0.025)
            volatility_factor[t] += rng.uniform(0.010, 0.050)

    # Asset betas to factors.
    betas = pd.DataFrame(0.0, index=asset_cols + hedge_cols, columns=["MARKET", "RATES", "COMMODITY", "VOL", "TREND"])

    betas.loc["US_EQ"] = [1.05, -0.15, 0.05, -0.10, 0.00]
    betas.loc["EU_EQ"] = [1.00, -0.15, 0.06, -0.10, 0.00]
    betas.loc["EM_EQ"] = [1.25, -0.10, 0.12, -0.15, 0.00]
    betas.loc["CREDIT"] = [0.55, -0.35, 0.02, -0.20, 0.00]
    betas.loc["REIT"] = [0.80, -0.55, 0.03, -0.10, 0.00]
    betas.loc["OIL"] = [0.30, 0.00, 1.15, -0.05, 0.00]
    betas.loc["COPPER"] = [0.45, 0.00, 0.95, -0.05, 0.00]
    betas.loc["GOLD"] = [-0.05, 0.30, 0.35, 0.10, 0.00]
    betas.loc["FX_CARRY"] = [0.35, -0.05, 0.10, -0.20, 0.00]
    betas.loc["TREND"] = [-0.20, 0.10, 0.05, 0.20, 1.00]
    betas.loc["BTC_PROXY"] = [1.15, -0.10, 0.10, -0.25, 0.00]

    betas.loc["INDEX_FUT"] = [1.02, -0.05, 0.02, -0.05, 0.00]
    betas.loc["TREASURY_FUT"] = [-0.25, 1.20, 0.05, 0.10, 0.00]
    betas.loc["GOLD_FUT"] = [-0.05, 0.25, 1.00, 0.10, 0.00]
    betas.loc["VIX_PROXY"] = [-2.20, 0.20, 0.00, 2.00, 0.20]

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07, "EU_EQ": 0.08, "EM_EQ": 0.13,
        "CREDIT": 0.04, "REIT": 0.11,
        "OIL": 0.20, "COPPER": 0.18, "GOLD": 0.12,
        "FX_CARRY": 0.07, "TREND": 0.08, "BTC_PROXY": 0.50,
        "INDEX_FUT": 0.03, "TREASURY_FUT": 0.025, "GOLD_FUT": 0.06, "VIX_PROXY": 0.35
    })

    annual_mu = pd.Series({
        "US_EQ": 0.07, "EU_EQ": 0.06, "EM_EQ": 0.08,
        "CREDIT": 0.045, "REIT": 0.06,
        "OIL": 0.045, "COPPER": 0.04, "GOLD": 0.035,
        "FX_CARRY": 0.03, "TREND": 0.05, "BTC_PROXY": 0.12,
        "INDEX_FUT": 0.055, "TREASURY_FUT": 0.025, "GOLD_FUT": 0.035, "VIX_PROXY": -0.10
    })

    factor_matrix = np.column_stack([market, rates, commodity, volatility_factor, trend_factor])
    factor_names = ["MARKET", "RATES", "COMMODITY", "VOL", "TREND"]

    returns = pd.DataFrame(index=dates, columns=asset_cols + hedge_cols, dtype=float)

    for col in returns.columns:
        systematic = factor_matrix @ betas.loc[col, factor_names].to_numpy()
        eps = idio_vol_ann[col] / np.sqrt(config.annualisation) * rng.standard_t(df=7, size=config.n_days) * np.sqrt((7 - 2) / 7)
        returns[col] = annual_mu[col] / config.annualisation + systematic + eps

    # Crypto crash idiosyncratic jumps.
    crypto_crash_mask = stress_type == "crypto_crash"
    returns.loc[crypto_crash_mask, "BTC_PROXY"] -= rng.uniform(0.12, 0.30, size=crypto_crash_mask.sum())

    # Index future tracks market with basis noise.
    market_series = pd.Series(market, index=dates, name="MARKET")
    returns["INDEX_FUT"] = market_series + 0.00001 + 0.003 * rng.standard_normal(config.n_days)

    # VIX proxy has positive convex behaviour on equity crash days.
    equity_crash_mask = stress_type == "equity_crash"
    returns.loc[equity_crash_mask, "VIX_PROXY"] += rng.uniform(0.08, 0.25, size=equity_crash_mask.sum())

    factors = pd.DataFrame({
        "date": dates,
        "MARKET": market,
        "RATES": rates,
        "COMMODITY": commodity,
        "VOL": volatility_factor,
        "TREND": trend_factor,
        "regime": regime,
        "stress_type": stress_type
    })

    returns_df = returns.copy()
    returns_df.insert(0, "date", dates)
    returns_df["MARKET"] = market
    returns_df["regime"] = regime
    returns_df["stress_type"] = stress_type

    metadata = pd.DataFrame({
        "instrument": asset_cols + hedge_cols,
        "instrument_type": ["portfolio_asset"] * len(asset_cols) + ["hedge_instrument"] * len(hedge_cols),
        "idio_vol_ann": [idio_vol_ann[x] for x in asset_cols + hedge_cols],
        "annual_mu": [annual_mu[x] for x in asset_cols + hedge_cols],
    }).merge(betas.reset_index().rename(columns={"index": "instrument"}), on="instrument", how="left")

    return returns_df, factors, metadata, asset_cols, hedge_cols


returns_df, factors, metadata, asset_cols, hedge_cols = simulate_beta_tail_hedge_universe(config)
returns = returns_df.set_index("date")[asset_cols + hedge_cols]
market_return = returns_df.set_index("date")["MARKET"]

returns_df.head(), metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["US_EQ", "EM_EQ", "CREDIT", "GOLD", "TREND", "BTC_PROXY", "INDEX_FUT", "VIX_PROXY"]:
    plt.plot(returns.index, (1 + returns[col]).cumprod(), label=col, alpha=0.8)
plt.title("Synthetic Asset and Hedge Instrument Growth")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(factors["date"], factors["regime"], drawstyle="steps-post")
plt.title("Volatility Regime")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 7. Build unhedged portfolio

We construct a diversified long-only portfolio.

This portfolio has meaningful market beta but also non-market exposures.

In [ ]:
portfolio_weights = pd.Series({
    "US_EQ": 0.18,
    "EU_EQ": 0.12,
    "EM_EQ": 0.10,
    "CREDIT": 0.12,
    "REIT": 0.08,
    "OIL": 0.06,
    "COPPER": 0.05,
    "GOLD": 0.10,
    "FX_CARRY": 0.06,
    "TREND": 0.09,
    "BTC_PROXY": 0.04
})

portfolio_weights = portfolio_weights / portfolio_weights.sum()

unhedged_return = returns[asset_cols] @ portfolio_weights

portfolio_weights.to_frame("weight")

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(unhedged_return.index, (1 + unhedged_return).cumprod(), label="unhedged portfolio")
plt.plot(market_return.index, (1 + market_return).cumprod(), label="market", alpha=0.7)
plt.title("Unhedged Portfolio vs Market")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 8. Beta estimation utilities

We estimate:

### Asset beta

$$
\beta_i = \frac{Cov(r_i,r_m)}{Var(r_m)}
$$

### Portfolio beta

$$
\beta_p = \sum_i w_i \beta_i
$$

### Downside beta

Estimated only on days where:

$$
r_m < q_{\alpha}(r_m)
$$

In [ ]:
def estimate_beta(asset_returns, market_returns):
    joined = pd.concat([asset_returns, market_returns.rename("MARKET")], axis=1).dropna()
    x = joined["MARKET"]

    betas = {}

    for col in asset_returns.columns:
        y = joined[col]
        var_m = x.var(ddof=1)
        beta = y.cov(x) / var_m if var_m > 0 else np.nan
        betas[col] = beta

    return pd.Series(betas)


def estimate_downside_beta(asset_returns, market_returns, quantile):
    threshold = market_returns.quantile(quantile)
    mask = market_returns <= threshold
    return estimate_beta(asset_returns.loc[mask], market_returns.loc[mask])


asset_betas = estimate_beta(returns[asset_cols + hedge_cols], market_return)
asset_downside_betas = estimate_downside_beta(returns[asset_cols + hedge_cols], market_return, config.downside_quantile)

beta_table = pd.DataFrame({
    "beta": asset_betas,
    "downside_beta": asset_downside_betas,
    "beta_minus_downside_beta": asset_betas - asset_downside_betas,
    "instrument_type": metadata.set_index("instrument")["instrument_type"]
})

beta_table.sort_values("downside_beta", ascending=False)

In [ ]:
portfolio_beta = float(portfolio_weights @ asset_betas[asset_cols])
portfolio_downside_beta = float(portfolio_weights @ asset_downside_betas[asset_cols])

pd.Series({
    "portfolio_beta": portfolio_beta,
    "portfolio_downside_beta": portfolio_downside_beta,
    "beta_weighted_exposure_usd": config.initial_capital * portfolio_beta,
    "downside_beta_weighted_exposure_usd": config.initial_capital * portfolio_downside_beta,
})

## 9. Rolling beta

Static beta can miss changing exposure.

Rolling beta:

$$
\beta_{p,t}^{roll} = \frac{Cov_t(R_p,R_m)} {Var_t(R_m)}
$$

over a trailing window.

In [ ]:
def rolling_beta(series, market, window):
    cov = series.rolling(window).cov(market)
    var = market.rolling(window).var()
    return cov / var


rolling_portfolio_beta = rolling_beta(unhedged_return, market_return, config.rolling_beta_window)
rolling_index_beta = rolling_beta(returns["INDEX_FUT"], market_return, config.rolling_beta_window)
rolling_vix_beta = rolling_beta(returns["VIX_PROXY"], market_return, config.rolling_beta_window)

rolling_beta_df = pd.DataFrame({
    "portfolio_beta": rolling_portfolio_beta,
    "index_future_beta": rolling_index_beta,
    "vix_proxy_beta": rolling_vix_beta
})

rolling_beta_df.tail()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(rolling_beta_df.index, rolling_beta_df["portfolio_beta"], label="portfolio beta")
plt.plot(rolling_beta_df.index, rolling_beta_df["index_future_beta"], label="index future beta")
plt.plot(rolling_beta_df.index, rolling_beta_df["vix_proxy_beta"], label="VIX proxy beta")
plt.axhline(0, linestyle="--")
plt.title("Rolling Betas to Market")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.show()

## 10. Beta hedge ratios

We calculate market futures hedge weights.

### Full beta hedge

$$
w_h = \frac{0-\beta_p}{\beta_h}
$$

### Partial beta hedge

$$
w_h = \frac{\beta^*-\beta_p}{\beta_h}
$$

### Downside beta hedge

Use downside betas instead of normal betas:

$$
w_h^{down} = \frac{\beta^{*,down}-\beta_p^{down}} {\beta_h^{down}}
$$

The downside hedge may be larger if portfolio downside beta exceeds normal beta.

In [ ]:
index_beta = float(asset_betas["INDEX_FUT"])
index_downside_beta = float(asset_downside_betas["INDEX_FUT"])

full_hedge_weight = (config.target_beta_full_hedge - portfolio_beta) / index_beta
partial_hedge_weight = (config.target_beta_partial_hedge - portfolio_beta) / index_beta
downside_full_hedge_weight = (config.target_beta_full_hedge - portfolio_downside_beta) / index_downside_beta

hedge_ratio_table = pd.DataFrame([
    {
        "hedge": "full_beta_hedge",
        "target_beta": config.target_beta_full_hedge,
        "portfolio_beta_used": portfolio_beta,
        "hedge_beta_used": index_beta,
        "index_future_weight": full_hedge_weight,
    },
    {
        "hedge": "partial_beta_hedge",
        "target_beta": config.target_beta_partial_hedge,
        "portfolio_beta_used": portfolio_beta,
        "hedge_beta_used": index_beta,
        "index_future_weight": partial_hedge_weight,
    },
    {
        "hedge": "downside_beta_full_hedge",
        "target_beta": config.target_beta_full_hedge,
        "portfolio_beta_used": portfolio_downside_beta,
        "hedge_beta_used": index_downside_beta,
        "index_future_weight": downside_full_hedge_weight,
    },
])

hedge_ratio_table

## 11. Futures overlay return engine

A hedge overlay has return:

$$
\begin{aligned}
R^{hedged}_t &= R^p_t \\
&\quad + w_h R^h_t \\
&\quad - cost_t \\
&\quad - carry_t
\end{aligned}
$$

Transaction cost is applied when the hedge changes.

Carry cost is a simple annualised drag for maintaining the hedge.

In [ ]:
def apply_static_futures_hedge(
    portfolio_return,
    hedge_return,
    hedge_weight,
    transaction_cost_bps,
    carry_bps_ann,
    annualisation,
):
    hedge_pnl = hedge_weight * hedge_return

    # One-time entry cost.
    cost = pd.Series(0.0, index=portfolio_return.index)
    cost.iloc[0] = abs(hedge_weight) * transaction_cost_bps / 10000.0

    # Carry drag proportional to absolute notional.
    carry = pd.Series(abs(hedge_weight) * carry_bps_ann / 10000.0 / annualisation, index=portfolio_return.index)

    hedged = portfolio_return + hedge_pnl - cost - carry

    return pd.DataFrame({
        "portfolio_return": portfolio_return,
        "hedge_return": hedge_return,
        "hedge_weight": hedge_weight,
        "hedge_pnl": hedge_pnl,
        "transaction_cost": cost,
        "carry_cost": carry,
        "net_return": hedged
    })


hedge_full = apply_static_futures_hedge(
    unhedged_return,
    returns["INDEX_FUT"],
    full_hedge_weight,
    config.transaction_cost_bps_futures,
    config.hedge_carry_bps_ann,
    config.annualisation
)

hedge_partial = apply_static_futures_hedge(
    unhedged_return,
    returns["INDEX_FUT"],
    partial_hedge_weight,
    config.transaction_cost_bps_futures,
    config.hedge_carry_bps_ann,
    config.annualisation
)

hedge_downside = apply_static_futures_hedge(
    unhedged_return,
    returns["INDEX_FUT"],
    downside_full_hedge_weight,
    config.transaction_cost_bps_futures,
    config.hedge_carry_bps_ann,
    config.annualisation
)

hedge_full.head()

## 12. Dynamic rolling beta hedge

A dynamic beta hedge uses trailing beta estimates.

At time $t$, hedge weight is:

$$
w_{h,t} = \frac{\beta^*-\hat\beta_{p,t-1}} {\hat\beta_{h,t-1}}
$$

This can adapt, but it adds turnover and can chase noisy beta estimates.

In [ ]:
def apply_dynamic_beta_hedge(
    portfolio_return,
    hedge_return,
    rolling_port_beta,
    rolling_hedge_beta,
    target_beta,
    transaction_cost_bps,
    carry_bps_ann,
    annualisation,
    max_abs_hedge=2.0
):
    beta_p = rolling_port_beta.shift(1)
    beta_h = rolling_hedge_beta.shift(1)

    hedge_weight = (target_beta - beta_p) / beta_h.replace(0, np.nan)
    hedge_weight = hedge_weight.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-max_abs_hedge, max_abs_hedge)

    hedge_pnl = hedge_weight * hedge_return

    turnover = hedge_weight.diff().abs().fillna(hedge_weight.abs())
    transaction_cost = turnover * transaction_cost_bps / 10000.0
    carry_cost = hedge_weight.abs() * carry_bps_ann / 10000.0 / annualisation

    net = portfolio_return + hedge_pnl - transaction_cost - carry_cost

    return pd.DataFrame({
        "portfolio_return": portfolio_return,
        "hedge_return": hedge_return,
        "hedge_weight": hedge_weight,
        "turnover": turnover,
        "hedge_pnl": hedge_pnl,
        "transaction_cost": transaction_cost,
        "carry_cost": carry_cost,
        "net_return": net
    })


dynamic_beta_hedge = apply_dynamic_beta_hedge(
    unhedged_return,
    returns["INDEX_FUT"],
    rolling_portfolio_beta,
    rolling_index_beta,
    config.target_beta_partial_hedge,
    config.transaction_cost_bps_futures,
    config.hedge_carry_bps_ann,
    config.annualisation
)

dynamic_beta_hedge.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(dynamic_beta_hedge.index, dynamic_beta_hedge["hedge_weight"])
plt.axhline(0, linestyle="--")
plt.title("Dynamic Rolling Beta Hedge Weight")
plt.xlabel("Date")
plt.ylabel("Index future weight")
plt.show()

## 13. Put-style convex tail hedge proxy

We build a simplified rolling put proxy on the market index.

For each day, approximate 21-day market return:

$$
R_{m,t:t+H}
$$

A put-like payoff over horizon $H$:

$$
Payoff_t = notional \times \max(K - (1 + R_{m,t:t+H}), 0)
$$

where:

$$
K = moneyness
$$

We distribute realised payoff back to the expiry date and subtract a steady premium budget.

This is not a full option pricer. It is a transparent convex hedge approximation.

In [ ]:
def rolling_put_proxy_returns(
    market_return,
    notional,
    maturity_days,
    moneyness,
    annual_budget,
    annualisation,
):
    cumulative_market = (1 + market_return).rolling(maturity_days).apply(lambda x: np.prod(x), raw=True)

    # Payoff arrives at the end of each rolling option horizon.
    payoff = notional * np.maximum(moneyness - cumulative_market, 0.0)
    payoff = payoff.fillna(0.0)

    # Premium budget paid daily.
    premium_cost = pd.Series(annual_budget / annualisation, index=market_return.index)

    net_return = payoff - premium_cost

    return pd.DataFrame({
        "market_horizon_growth": cumulative_market,
        "put_payoff_return": payoff,
        "premium_cost": premium_cost,
        "net_put_proxy_return": net_return
    })


put_proxy = rolling_put_proxy_returns(
    market_return,
    notional=1.0,
    maturity_days=config.put_maturity_days,
    moneyness=config.put_moneyness,
    annual_budget=config.put_budget_ann,
    annualisation=config.annualisation
)

# Use a fixed allocation to the put proxy as a return overlay.
put_notional_weight = 0.40
put_hedged_return = unhedged_return + put_notional_weight * put_proxy["net_put_proxy_return"]

put_proxy.head()

## 14. Collar-style hedge proxy

A collar buys downside protection and sells upside.

Simplified collar return:

$$
\begin{aligned}
collar &= put\_payoff \\
&\quad - call\_giveup \\
&\quad - net\_premium
\end{aligned}
$$

where:

$$
call\_giveup = \max((1+R_{m,t:t+H})-K_{call},0)
$$

This lowers hedge cost but gives away upside.

In [ ]:
def rolling_collar_proxy_returns(
    market_return,
    notional,
    maturity_days,
    put_moneyness,
    call_moneyness,
    annual_budget,
    annualisation,
):
    cumulative_market = (1 + market_return).rolling(maturity_days).apply(lambda x: np.prod(x), raw=True)

    put_payoff = notional * np.maximum(put_moneyness - cumulative_market, 0.0)
    call_giveup = notional * np.maximum(cumulative_market - call_moneyness, 0.0)

    premium_cost = pd.Series(annual_budget / annualisation, index=market_return.index)

    net = put_payoff - call_giveup - premium_cost

    return pd.DataFrame({
        "market_horizon_growth": cumulative_market,
        "put_payoff_return": put_payoff.fillna(0.0),
        "call_giveup_return": call_giveup.fillna(0.0),
        "premium_cost": premium_cost,
        "net_collar_proxy_return": net.fillna(-premium_cost)
    })


collar_proxy = rolling_collar_proxy_returns(
    market_return,
    notional=1.0,
    maturity_days=config.put_maturity_days,
    put_moneyness=config.put_moneyness,
    call_moneyness=config.collar_call_moneyness,
    annual_budget=config.put_budget_ann * 0.35,
    annualisation=config.annualisation
)

collar_notional_weight = 0.40
collar_hedged_return = unhedged_return + collar_notional_weight * collar_proxy["net_collar_proxy_return"]

collar_proxy.head()

## 15. Multi-hedge overlay

A practical tail hedge may combine:

1. partial beta hedge with index futures;
2. treasury future;
3. gold future;
4. VIX proxy;
5. put proxy.

The goal is not to eliminate all risk.

The goal is to reduce left-tail losses while controlling bleed.

In [ ]:
multi_hedge_weights = pd.Series({
    "INDEX_FUT": partial_hedge_weight,
    "TREASURY_FUT": 0.20,
    "GOLD_FUT": 0.08,
    "VIX_PROXY": 0.05,
})

def apply_multi_hedge_overlay(
    portfolio_return,
    hedge_returns,
    hedge_weights,
    transaction_cost_bps,
    carry_bps_ann,
    annualisation,
):
    hedge_pnl_by_instrument = hedge_returns[hedge_weights.index].multiply(hedge_weights, axis=1)
    hedge_pnl = hedge_pnl_by_instrument.sum(axis=1)

    # One-time entry cost and daily carry.
    transaction_cost = pd.Series(0.0, index=portfolio_return.index)
    transaction_cost.iloc[0] = hedge_weights.abs().sum() * transaction_cost_bps / 10000.0

    carry_cost = pd.Series(hedge_weights.abs().sum() * carry_bps_ann / 10000.0 / annualisation, index=portfolio_return.index)

    net = portfolio_return + hedge_pnl - transaction_cost - carry_cost

    out = pd.DataFrame({
        "portfolio_return": portfolio_return,
        "hedge_pnl": hedge_pnl,
        "transaction_cost": transaction_cost,
        "carry_cost": carry_cost,
        "net_return": net
    })

    for col in hedge_pnl_by_instrument.columns:
        out[f"{col}_pnl"] = hedge_pnl_by_instrument[col]

    return out


multi_hedge = apply_multi_hedge_overlay(
    unhedged_return,
    returns[hedge_cols],
    multi_hedge_weights,
    config.transaction_cost_bps_futures,
    config.hedge_carry_bps_ann,
    config.annualisation
)

multi_hedge.head()

## 16. Collect hedge strategy returns

We compare:

1. unhedged;
2. full static beta hedge;
3. partial static beta hedge;
4. downside-beta hedge;
5. dynamic rolling beta hedge;
6. put proxy hedge;
7. collar proxy hedge;
8. multi-hedge overlay.

In [ ]:
strategy_returns = pd.DataFrame({
    "unhedged": unhedged_return,
    "full_beta_hedge": hedge_full["net_return"],
    "partial_beta_hedge": hedge_partial["net_return"],
    "downside_beta_hedge": hedge_downside["net_return"],
    "dynamic_beta_hedge": dynamic_beta_hedge["net_return"],
    "put_proxy_tail_hedge": put_hedged_return,
    "collar_proxy_tail_hedge": collar_hedged_return,
    "multi_hedge_overlay": multi_hedge["net_return"],
})

strategy_returns.head()

## 17. Performance and drawdown metrics

We evaluate:

- annualised return;
- annualised volatility;
- Sharpe proxy;
- skew;
- kurtosis;
- max drawdown;
- worst day;
- beta to market;
- downside beta to market.

In [ ]:
def max_drawdown_from_returns(r):
    nav = (1 + r).cumprod()
    dd = nav / nav.cummax() - 1
    return dd, float(dd.min())


def strategy_diagnostics(strategy_returns, market_return):
    rows = []

    for name in strategy_returns.columns:
        r = strategy_returns[name].dropna()
        beta = estimate_beta(r.to_frame(name), market_return.reindex(r.index))[name]
        downside_beta = estimate_downside_beta(r.to_frame(name), market_return.reindex(r.index), config.downside_quantile)[name]
        dd, mdd = max_drawdown_from_returns(r)

        rows.append({
            "strategy": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(r.mean() / r.std(ddof=1) * np.sqrt(config.annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "skew": float(r.skew()),
            "excess_kurtosis": float(r.kurtosis()),
            "max_drawdown": mdd,
            "worst_day": float(r.min()),
            "beta": float(beta),
            "downside_beta": float(downside_beta),
            "total_return": float((1 + r).prod() - 1),
        })

    return pd.DataFrame(rows).sort_values("max_drawdown", ascending=False)


strategy_summary = strategy_diagnostics(strategy_returns, market_return)
strategy_summary

In [ ]:
plt.figure(figsize=(12, 6))
for col in strategy_returns.columns:
    plt.plot(strategy_returns.index, (1 + strategy_returns[col]).cumprod(), label=col, alpha=0.85)
plt.title("Hedged Strategy Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=2)
plt.show()

plt.figure(figsize=(10, 6))
plt.barh(strategy_summary.sort_values("downside_beta")["strategy"], strategy_summary.sort_values("downside_beta")["downside_beta"])
plt.axvline(0, linestyle="--")
plt.title("Downside Beta by Hedge Strategy")
plt.xlabel("Downside beta")
plt.ylabel("Strategy")
plt.show()

## 18. VaR and CVaR before and after hedging

We reuse one-day historical VaR/CVaR:

$$
VaR_\alpha = quantile_\alpha(-R)
$$

$$
CVaR_\alpha = E[-R \mid -R \ge VaR_\alpha]
$$

In [ ]:
def historical_var_cvar(losses, alpha):
    losses = np.asarray(losses, dtype=float)
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    cvar = tail.mean() if len(tail) else np.nan
    return float(var), float(cvar)


tail_rows = []

for strategy in strategy_returns.columns:
    losses = -strategy_returns[strategy].dropna()

    for alpha in [0.95, 0.99]:
        var, cvar = historical_var_cvar(losses, alpha)

        tail_rows.append({
            "strategy": strategy,
            "alpha": alpha,
            "VaR": var,
            "CVaR": cvar,
            "tail_ratio_CVaR_to_VaR": cvar / var if var != 0 else np.nan
        })

tail_risk_table = pd.DataFrame(tail_rows)
tail_risk_table.sort_values(["alpha", "CVaR"])

In [ ]:
sub = tail_risk_table[tail_risk_table["alpha"] == 0.99].sort_values("CVaR")

plt.figure(figsize=(10, 6))
plt.barh(sub["strategy"], sub["CVaR"])
plt.title("99% Historical CVaR by Hedge Strategy")
plt.xlabel("One-day CVaR loss")
plt.ylabel("Strategy")
plt.show()

## 19. Hedge effectiveness metrics

Define hedge effectiveness for risk measure $M$:

$$
Effectiveness = 1 - \frac{M_{hedged}}{M_{unhedged}}
$$

A positive value means risk reduction.

A negative value means the hedge made the metric worse.

In [ ]:
def hedge_effectiveness(tail_risk_table, metric="CVaR", alpha=0.99):
    base = tail_risk_table[
        (tail_risk_table["strategy"] == "unhedged")
        & (tail_risk_table["alpha"] == alpha)
    ][metric].iloc[0]

    rows = []

    for _, row in tail_risk_table[tail_risk_table["alpha"] == alpha].iterrows():
        rows.append({
            "strategy": row["strategy"],
            "alpha": alpha,
            "metric": metric,
            "metric_value": row[metric],
            "unhedged_metric": base,
            "hedge_effectiveness": 1 - row[metric] / base if base != 0 else np.nan
        })

    return pd.DataFrame(rows).sort_values("hedge_effectiveness", ascending=False)


hedge_effectiveness_cvar99 = hedge_effectiveness(tail_risk_table, metric="CVaR", alpha=0.99)
hedge_effectiveness_cvar99

## 20. Stress scenario hedge performance

We build hypothetical stress scenarios:

1. equity crash;
2. inflation shock;
3. commodity crash;
4. crypto crash;
5. rates shock;
6. volatility spike.

Then we apply each hedge overlay to the stress shock.

In [ ]:
def build_stress_scenarios(asset_cols, hedge_cols):
    cols = asset_cols + hedge_cols + ["MARKET"]

    scenarios = {
        "equity_crash": {
            "MARKET": -0.08,
            "US_EQ": -0.085, "EU_EQ": -0.090, "EM_EQ": -0.120,
            "CREDIT": -0.045, "REIT": -0.080,
            "OIL": -0.040, "COPPER": -0.060, "GOLD": 0.015,
            "FX_CARRY": -0.030, "TREND": 0.025, "BTC_PROXY": -0.200,
            "INDEX_FUT": -0.080, "TREASURY_FUT": 0.020, "GOLD_FUT": 0.018, "VIX_PROXY": 0.280,
        },
        "inflation_shock": {
            "MARKET": -0.035,
            "US_EQ": -0.040, "EU_EQ": -0.045, "EM_EQ": -0.060,
            "CREDIT": -0.040, "REIT": -0.070,
            "OIL": 0.100, "COPPER": 0.075, "GOLD": 0.040,
            "FX_CARRY": 0.005, "TREND": 0.020, "BTC_PROXY": -0.080,
            "INDEX_FUT": -0.035, "TREASURY_FUT": -0.045, "GOLD_FUT": 0.040, "VIX_PROXY": 0.080,
        },
        "commodity_crash": {
            "MARKET": -0.020,
            "US_EQ": -0.020, "EU_EQ": -0.025, "EM_EQ": -0.040,
            "CREDIT": -0.015, "REIT": -0.020,
            "OIL": -0.130, "COPPER": -0.110, "GOLD": -0.020,
            "FX_CARRY": -0.020, "TREND": 0.025, "BTC_PROXY": -0.070,
            "INDEX_FUT": -0.020, "TREASURY_FUT": 0.010, "GOLD_FUT": -0.020, "VIX_PROXY": 0.050,
        },
        "crypto_crash": {
            "MARKET": -0.015,
            "US_EQ": -0.015, "EU_EQ": -0.015, "EM_EQ": -0.020,
            "CREDIT": -0.010, "REIT": -0.010,
            "OIL": -0.005, "COPPER": -0.005, "GOLD": 0.005,
            "FX_CARRY": -0.005, "TREND": 0.005, "BTC_PROXY": -0.350,
            "INDEX_FUT": -0.015, "TREASURY_FUT": 0.004, "GOLD_FUT": 0.006, "VIX_PROXY": 0.070,
        },
        "rates_shock": {
            "MARKET": -0.030,
            "US_EQ": -0.035, "EU_EQ": -0.035, "EM_EQ": -0.045,
            "CREDIT": -0.050, "REIT": -0.090,
            "OIL": -0.010, "COPPER": -0.015, "GOLD": -0.020,
            "FX_CARRY": 0.005, "TREND": 0.000, "BTC_PROXY": -0.080,
            "INDEX_FUT": -0.030, "TREASURY_FUT": -0.050, "GOLD_FUT": -0.020, "VIX_PROXY": 0.060,
        },
        "volatility_spike": {
            "MARKET": -0.045,
            "US_EQ": -0.050, "EU_EQ": -0.052, "EM_EQ": -0.070,
            "CREDIT": -0.035, "REIT": -0.045,
            "OIL": -0.020, "COPPER": -0.030, "GOLD": 0.010,
            "FX_CARRY": -0.030, "TREND": 0.020, "BTC_PROXY": -0.120,
            "INDEX_FUT": -0.045, "TREASURY_FUT": 0.018, "GOLD_FUT": 0.012, "VIX_PROXY": 0.220,
        },
    }

    return pd.DataFrame(scenarios).T.reindex(columns=cols).fillna(0.0)


stress_scenarios = build_stress_scenarios(asset_cols, hedge_cols)
stress_scenarios

In [ ]:
def stress_strategy_returns(stress_scenarios, portfolio_weights, hedge_weights, put_weight, collar_weight):
    rows = []

    for scenario, shock in stress_scenarios.iterrows():
        base_return = float(portfolio_weights @ shock[asset_cols])

        full_return = base_return + full_hedge_weight * shock["INDEX_FUT"]
        partial_return = base_return + partial_hedge_weight * shock["INDEX_FUT"]
        downside_return = base_return + downside_full_hedge_weight * shock["INDEX_FUT"]

        # Dynamic approximated as partial hedge in scenario.
        dynamic_return = partial_return

        # Put/collar proxies approximate convex payout on market scenario.
        market_growth = 1 + shock["MARKET"]
        put_payoff = max(config.put_moneyness - market_growth, 0.0) - config.put_budget_ann / config.annualisation
        call_giveup = max(market_growth - config.collar_call_moneyness, 0.0)
        collar_payoff = max(config.put_moneyness - market_growth, 0.0) - call_giveup - 0.35 * config.put_budget_ann / config.annualisation

        put_return = base_return + put_weight * put_payoff
        collar_return = base_return + collar_weight * collar_payoff

        multi_return = base_return + float(hedge_weights @ shock[hedge_weights.index])

        strategy_map = {
            "unhedged": base_return,
            "full_beta_hedge": full_return,
            "partial_beta_hedge": partial_return,
            "downside_beta_hedge": downside_return,
            "dynamic_beta_hedge": dynamic_return,
            "put_proxy_tail_hedge": put_return,
            "collar_proxy_tail_hedge": collar_return,
            "multi_hedge_overlay": multi_return,
        }

        for strategy, ret in strategy_map.items():
            rows.append({
                "scenario": scenario,
                "strategy": strategy,
                "scenario_return": ret,
                "scenario_loss": -ret
            })

    return pd.DataFrame(rows)


stress_strategy = stress_strategy_returns(
    stress_scenarios,
    portfolio_weights,
    multi_hedge_weights,
    put_notional_weight,
    collar_notional_weight
)

stress_strategy.sort_values("scenario_loss", ascending=False).head(15)

In [ ]:
plt.figure(figsize=(12, 6))
for strategy in ["unhedged", "partial_beta_hedge", "put_proxy_tail_hedge", "multi_hedge_overlay"]:
    sub = stress_strategy[stress_strategy["strategy"] == strategy]
    plt.plot(sub["scenario"], sub["scenario_loss"], marker="o", label=strategy)
plt.xticks(rotation=45, ha="right")
plt.title("Stress Scenario Loss by Hedge Strategy")
plt.ylabel("Scenario loss")
plt.legend()
plt.tight_layout()
plt.show()

## 21. Hedge attribution

A hedge may improve tail risk while reducing average return.

We attribute total return into:

1. base portfolio return;
2. hedge PnL;
3. transaction costs;
4. carry / premium costs.

In [ ]:
hedge_attribution = pd.DataFrame([
    {
        "strategy": "full_beta_hedge",
        "base_return_ann": hedge_full["portfolio_return"].mean() * config.annualisation,
        "hedge_pnl_ann": hedge_full["hedge_pnl"].mean() * config.annualisation,
        "cost_ann": -(hedge_full["transaction_cost"].mean() + hedge_full["carry_cost"].mean()) * config.annualisation,
        "net_return_ann": hedge_full["net_return"].mean() * config.annualisation,
    },
    {
        "strategy": "partial_beta_hedge",
        "base_return_ann": hedge_partial["portfolio_return"].mean() * config.annualisation,
        "hedge_pnl_ann": hedge_partial["hedge_pnl"].mean() * config.annualisation,
        "cost_ann": -(hedge_partial["transaction_cost"].mean() + hedge_partial["carry_cost"].mean()) * config.annualisation,
        "net_return_ann": hedge_partial["net_return"].mean() * config.annualisation,
    },
    {
        "strategy": "downside_beta_hedge",
        "base_return_ann": hedge_downside["portfolio_return"].mean() * config.annualisation,
        "hedge_pnl_ann": hedge_downside["hedge_pnl"].mean() * config.annualisation,
        "cost_ann": -(hedge_downside["transaction_cost"].mean() + hedge_downside["carry_cost"].mean()) * config.annualisation,
        "net_return_ann": hedge_downside["net_return"].mean() * config.annualisation,
    },
    {
        "strategy": "dynamic_beta_hedge",
        "base_return_ann": dynamic_beta_hedge["portfolio_return"].mean() * config.annualisation,
        "hedge_pnl_ann": dynamic_beta_hedge["hedge_pnl"].mean() * config.annualisation,
        "cost_ann": -(dynamic_beta_hedge["transaction_cost"].mean() + dynamic_beta_hedge["carry_cost"].mean()) * config.annualisation,
        "net_return_ann": dynamic_beta_hedge["net_return"].mean() * config.annualisation,
    },
    {
        "strategy": "multi_hedge_overlay",
        "base_return_ann": multi_hedge["portfolio_return"].mean() * config.annualisation,
        "hedge_pnl_ann": multi_hedge["hedge_pnl"].mean() * config.annualisation,
        "cost_ann": -(multi_hedge["transaction_cost"].mean() + multi_hedge["carry_cost"].mean()) * config.annualisation,
        "net_return_ann": multi_hedge["net_return"].mean() * config.annualisation,
    },
    {
        "strategy": "put_proxy_tail_hedge",
        "base_return_ann": unhedged_return.mean() * config.annualisation,
        "hedge_pnl_ann": (put_notional_weight * put_proxy["put_payoff_return"]).mean() * config.annualisation,
        "cost_ann": -(put_notional_weight * put_proxy["premium_cost"]).mean() * config.annualisation,
        "net_return_ann": put_hedged_return.mean() * config.annualisation,
    },
])

hedge_attribution

In [ ]:
plot_cols = ["base_return_ann", "hedge_pnl_ann", "cost_ann"]
plot_df = hedge_attribution.set_index("strategy")[plot_cols]

plt.figure(figsize=(12, 6))
bottom = np.zeros(len(plot_df))
x = np.arange(len(plot_df))

for col in plot_cols:
    plt.bar(x, plot_df[col], bottom=bottom, label=col)
    bottom += plot_df[col].values

plt.axhline(0, linestyle="--")
plt.xticks(x, plot_df.index, rotation=45, ha="right")
plt.title("Annualised Hedge Attribution")
plt.ylabel("Annualised return contribution")
plt.legend()
plt.tight_layout()
plt.show()

## 22. Crisis-period hedge effectiveness

A hedge should be judged in the periods where protection is needed.

We isolate stress days and compare average losses.

In [ ]:
stress_mask = returns_df.set_index("date")["stress_type"] != "normal"
crisis_period_report = []

for strategy in strategy_returns.columns:
    r = strategy_returns[strategy]

    crisis_period_report.append({
        "strategy": strategy,
        "n_stress_days": int(stress_mask.sum()),
        "mean_return_normal_ann": float(r[~stress_mask].mean() * config.annualisation),
        "mean_return_stress_ann": float(r[stress_mask].mean() * config.annualisation),
        "worst_stress_day": float(r[stress_mask].min()),
        "stress_day_hit_rate_positive": float((r[stress_mask] > 0).mean()),
    })

crisis_period_report = pd.DataFrame(crisis_period_report).sort_values("worst_stress_day", ascending=False)
crisis_period_report

## 23. Contract sizing example

If an index future price is $F$, multiplier is $M$, and portfolio value is $V$:

$$
contracts = \frac{w_h V}{F M}
$$

A negative number means short futures.

This translates hedge weight into implementation units.

In [ ]:
def futures_contract_sizing(hedge_weight, portfolio_value, futures_price, multiplier):
    notional = hedge_weight * portfolio_value
    contracts = notional / (futures_price * multiplier)
    return {
        "hedge_weight": hedge_weight,
        "portfolio_value": portfolio_value,
        "futures_price": futures_price,
        "multiplier": multiplier,
        "hedge_notional": notional,
        "contracts": contracts
    }


contract_examples = pd.DataFrame([
    futures_contract_sizing(full_hedge_weight, config.initial_capital, futures_price=5000.0, multiplier=50.0) | {"hedge": "full_beta_hedge"},
    futures_contract_sizing(partial_hedge_weight, config.initial_capital, futures_price=5000.0, multiplier=50.0) | {"hedge": "partial_beta_hedge"},
    futures_contract_sizing(downside_full_hedge_weight, config.initial_capital, futures_price=5000.0, multiplier=50.0) | {"hedge": "downside_beta_hedge"},
])

contract_examples

## 24. Hedge basis risk

Beta hedges can fail because the hedge instrument is not the portfolio.

Basis residual:

$$
basis_t = R^p_t - \beta_p R^h_t
$$

Large basis volatility means the hedge instrument is an imperfect proxy.

In [ ]:
basis_residual = unhedged_return - portfolio_beta * returns["INDEX_FUT"]
basis_report = pd.Series({
    "basis_mean_ann": basis_residual.mean() * config.annualisation,
    "basis_vol_ann": basis_residual.std(ddof=1) * np.sqrt(config.annualisation),
    "basis_worst_day": basis_residual.min(),
    "basis_corr_with_market": basis_residual.corr(market_return),
})

basis_report

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(basis_residual.index, basis_residual.rolling(21).mean())
plt.axhline(0, linestyle="--")
plt.title("Beta Hedge Basis Residual, 21-Day Average")
plt.xlabel("Date")
plt.ylabel("Residual return")
plt.show()

## 25. Governance flags

A hedge programme should track:

1. residual beta;
2. downside beta;
3. CVaR reduction;
4. hedge cost;
5. stress loss reduction;
6. basis risk;
7. dynamic hedge turnover.

These flags turn hedging into a controlled process instead of a vibes-based overlay.

In [ ]:
unhedged_cvar99 = tail_risk_table[
    (tail_risk_table["strategy"] == "unhedged")
    & (tail_risk_table["alpha"] == 0.99)
]["CVaR"].iloc[0]

governance_rows = []

for strategy in strategy_returns.columns:
    summary = strategy_summary[strategy_summary["strategy"] == strategy].iloc[0]
    cvar99 = tail_risk_table[
        (tail_risk_table["strategy"] == strategy)
        & (tail_risk_table["alpha"] == 0.99)
    ]["CVaR"].iloc[0]

    cvar_reduction = 1 - cvar99 / unhedged_cvar99 if unhedged_cvar99 != 0 else np.nan

    worst_stress = stress_strategy[stress_strategy["strategy"] == strategy]["scenario_loss"].max()

    governance_rows.append({
        "strategy": strategy,
        "beta": summary["beta"],
        "downside_beta": summary["downside_beta"],
        "cvar99": cvar99,
        "cvar99_reduction_vs_unhedged": cvar_reduction,
        "max_drawdown": summary["max_drawdown"],
        "worst_scenario_loss": worst_stress,
        "flag_downside_beta_above_0_5": bool(summary["downside_beta"] > 0.5),
        "flag_cvar_reduction_below_10pct": bool(cvar_reduction < 0.10 and strategy != "unhedged"),
        "flag_worst_scenario_loss_above_8pct": bool(worst_stress > 0.08),
    })

governance_flags = pd.DataFrame(governance_rows).sort_values("cvar99_reduction_vs_unhedged", ascending=False)
governance_flags

## 26. Practical checklist for beta-weighted tail hedging

Before trusting a hedge:

1. **Estimate normal beta**  
   What is the portfolio's average market exposure?

2. **Estimate downside beta**  
   Does beta rise in market sell-offs?

3. **Check hedge beta**  
   Is the hedge instrument actually close to the intended beta?

4. **Calculate beta-weighted exposure**  
   How many dollars of market-equivalent risk exist?

5. **Size futures hedge**  
   What notional or contracts are needed?

6. **Measure basis risk**  
   Does the portfolio move differently from the hedge?

7. **Compare VaR/CVaR**  
   Does the hedge reduce tail risk?

8. **Stress test**  
   Does it work in the scenarios that matter?

9. **Track cost and bleed**  
   Is protection too expensive in calm markets?

10. **Avoid false comfort**  
   A beta hedge is not a complete crash hedge.

## 27. Saving outputs

The notebook saves:

1. synthetic returns;
2. factor returns;
3. metadata;
4. portfolio weights;
5. beta table;
6. rolling beta;
7. hedge ratio table;
8. static hedge returns;
9. dynamic hedge returns;
10. put and collar proxy returns;
11. multi-hedge overlay;
12. strategy returns;
13. strategy diagnostics;
14. tail-risk table;
15. hedge effectiveness;
16. stress scenarios;
17. stress strategy results;
18. hedge attribution;
19. crisis-period report;
20. contract examples;
21. basis-risk report;
22. governance flags;
23. manifest.

In [ ]:
output_dir = Path("data/processed/beta_weighted_tail_risk_hedging_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
factors_path = output_dir / "factors.csv"
metadata_path = output_dir / "metadata.csv"
portfolio_weights_path = output_dir / "portfolio_weights.csv"
beta_table_path = output_dir / "beta_table.csv"
rolling_beta_path = output_dir / "rolling_beta.csv"
hedge_ratio_path = output_dir / "hedge_ratio_table.csv"
hedge_full_path = output_dir / "hedge_full.csv"
hedge_partial_path = output_dir / "hedge_partial.csv"
hedge_downside_path = output_dir / "hedge_downside.csv"
dynamic_hedge_path = output_dir / "dynamic_beta_hedge.csv"
put_proxy_path = output_dir / "put_proxy.csv"
collar_proxy_path = output_dir / "collar_proxy.csv"
multi_hedge_path = output_dir / "multi_hedge.csv"
strategy_returns_path = output_dir / "strategy_returns.csv"
strategy_summary_path = output_dir / "strategy_summary.csv"
tail_risk_path = output_dir / "tail_risk_table.csv"
hedge_effectiveness_path = output_dir / "hedge_effectiveness_cvar99.csv"
stress_scenarios_path = output_dir / "stress_scenarios.csv"
stress_strategy_path = output_dir / "stress_strategy_results.csv"
hedge_attribution_path = output_dir / "hedge_attribution.csv"
crisis_period_path = output_dir / "crisis_period_report.csv"
contract_examples_path = output_dir / "contract_examples.csv"
basis_report_path = output_dir / "basis_report.csv"
governance_flags_path = output_dir / "governance_flags.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
factors.to_csv(factors_path, index=False)
metadata.to_csv(metadata_path, index=False)
portfolio_weights.to_frame("weight").to_csv(portfolio_weights_path)
beta_table.to_csv(beta_table_path)
rolling_beta_df.to_csv(rolling_beta_path)
hedge_ratio_table.to_csv(hedge_ratio_path, index=False)
hedge_full.to_csv(hedge_full_path)
hedge_partial.to_csv(hedge_partial_path)
hedge_downside.to_csv(hedge_downside_path)
dynamic_beta_hedge.to_csv(dynamic_hedge_path)
put_proxy.to_csv(put_proxy_path)
collar_proxy.to_csv(collar_proxy_path)
multi_hedge.to_csv(multi_hedge_path)
strategy_returns.to_csv(strategy_returns_path)
strategy_summary.to_csv(strategy_summary_path, index=False)
tail_risk_table.to_csv(tail_risk_path, index=False)
hedge_effectiveness_cvar99.to_csv(hedge_effectiveness_path, index=False)
stress_scenarios.to_csv(stress_scenarios_path)
stress_strategy.to_csv(stress_strategy_path, index=False)
hedge_attribution.to_csv(hedge_attribution_path, index=False)
crisis_period_report.to_csv(crisis_period_path, index=False)
contract_examples.to_csv(contract_examples_path, index=False)
basis_report.to_frame("value").to_csv(basis_report_path)
governance_flags.to_csv(governance_flags_path, index=False)

manifest = {
    "dataset_name": "beta_weighted_tail_risk_hedging_outputs",
    "schema_version": "beta_weighted_tail_risk_hedging_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "hedge_cols": hedge_cols,
    "portfolio_beta": portfolio_beta,
    "portfolio_downside_beta": portfolio_downside_beta,
    "hedge_ratio_table": hedge_ratio_table.to_dict(orient="records"),
    "strategy_summary": strategy_summary.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "notes": [
        "Beta-weighted exposure is calculated using market beta and portfolio value.",
        "Normal beta and downside beta are compared.",
        "Static full, partial, and downside beta hedges use index futures.",
        "Dynamic beta hedge uses trailing rolling beta estimates shifted by one day.",
        "Put and collar hedges are simplified convex payoff proxies, not full option pricing models.",
        "Multi-hedge overlay combines index, treasury, gold, and VIX-style hedge instruments.",
        "Hedge effectiveness is measured using CVaR reduction, stress scenario loss, drawdown, and hedge cost."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, beta_table_path, strategy_summary_path, governance_flags_path, manifest_path

## 28. Limitations

### 28.1 Synthetic data

This notebook uses synthetic returns and hedge instruments.

Real hedging requires live positions, exact contract specs, liquidity, margin, and execution costs.

### 28.2 Linear beta limitation

Beta hedging assumes a linear relationship.

Tail losses can be nonlinear, path-dependent, or driven by non-market factors.

### 28.3 Downside beta estimation error

Tail beta uses fewer observations, so it is noisy.

### 28.4 Basis risk

Index futures may not track the portfolio exactly.

Sector, region, credit, rates, commodity, and FX exposures create residual risk.

### 28.5 Simplified option proxy

The put and collar proxies are not option pricing models.

They ignore implied volatility, delta, gamma, vega, theta, skew, transaction costs, and roll timing.

### 28.6 Static hedge weights

Static hedges can become stale.

Dynamic hedges add turnover and estimation noise.

### 28.7 Stress scenarios are simplified

Real stress scenarios should be approved, documented, and reviewed by risk governance.

## 29. Research frontier and extensions

Important extensions include:

1. **Option-based tail hedging with Greeks**  
   Use real option prices, delta, gamma, vega, theta, and skew.

2. **Delta-gamma VaR**  
   Include nonlinear derivative exposures.

3. **Robust beta estimation**  
   Use shrinkage, Bayesian beta, and state-space beta.

4. **Regime-switching hedge ratios**  
   Estimate different betas in calm and crisis regimes.

5. **CVaR-minimising hedge optimisation**  
   Choose hedge weights directly by minimising Expected Shortfall.

6. **Hedge carry optimisation**  
   Balance protection versus bleed.

7. **Basis-risk model**  
   Decompose hedge error into sector, style, and asset-class residuals.

8. **Dynamic option rolling**  
   Systematically roll put spreads or collars.

9. **Liquidity stress hedging**  
   Include spread widening and futures margin changes.

10. **Chinese futures tail hedging**  
   Use commodity futures, night-session volatility, price limits, and cross-contract hedge ratios.

## 30. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_08_risk_parity_and_equal_risk_contribution**  
   Allocate capital by risk contribution.

2. **04_09_black_litterman_portfolio_construction**  
   Blend priors and views for allocation.

3. **04_10_portfolio_constraints_margin_and_leverage**  
   Add real implementation constraints.

4. **05_03_transaction_costs_slippage_latency**  
   Model hedge execution costs.

5. **06_08_real_time_risk_dashboard**  
   Monitor beta, CVaR, stress losses, and hedge effectiveness.

6. **07_02_options_tail_hedging_capstone**  
   Build a full option-based tail hedge engine.

## 31. Summary

This notebook implemented beta-weighted tail-risk hedging.

It showed how to:

1. simulate a portfolio and hedge instruments;
2. estimate normal beta and downside beta;
3. calculate beta-weighted exposure;
4. size full, partial, and downside-beta futures hedges;
5. implement static futures overlays;
6. implement a dynamic rolling beta hedge;
7. build simplified put and collar tail-hedge proxies;
8. combine hedges into a multi-asset overlay;
9. compare return, volatility, beta, downside beta, drawdown, VaR, and CVaR;
10. stress test hedge strategies;
11. attribute hedge PnL, carry, and cost;
12. analyse basis risk;
13. convert hedge notional into futures contracts;
14. create governance flags and saved reports.

The key computational takeaway:

> Hedging is an overlay accounting problem: estimate exposure, size hedge instruments, apply costs, and measure residual risk.

The key financial takeaway:

> A beta hedge can reduce market exposure, but only convex or diversified tail hedges can address crash losses, basis risk, and nonlinear downside.

## 32. Further reading

- Grinold and Kahn, *Active Portfolio Management.*
- Jorion, *Value at Risk.*
- McNeil, Frey, and Embrechts, *Quantitative Risk Management.*
- Literature on beta hedging, downside beta, crisis alpha, tail-risk hedging, protective puts, collars, and hedge effectiveness.
- CBOE and option strategy literature on put protection and collars.